# dk EGAT 모델 풀 학습 (Colab A100)

> ⚠️ **셀 5(학습)가 이미 완료돼 있습니다 — 아래 5)번 백업 셀부터 먼저 실행하세요.** 세션이 끊기면
> `results/dk_gat*.pt`가 전부 사라집니다. 백업 후에 이어서 새 커맨드(v2)로 재실행하면 됩니다.

> **최종 업데이트**: 2026-07-14 (**Entity-Pair Graph 추가** — A→B,B→C⇒A→C 합성 추론을 직접
> 겨냥, `dk_gat_v2`에 반영): 지금까지 dk_gat의 Entity+Sentence GAT는 엔티티 임베딩만 보강할 뿐,
> "쌍 (a,b)가 지금 어떤 관계로 예측되는지"를 다른 쌍이 직접 읽는 경로가 없었음. **두 번째 그래프
> 단계**(`--use_pair_graph`)를 추가 — 노드 = (h,t) 쌍 자체(엔티티 아님), `same-head`/
> `same-tail`/`bridge-succ`/`bridge-pred` 4타입 방향성 엣지로 연결하고, 이 쌍의 provisional
> 관계 예측(logits)까지 노드 특징에 얹어서(관계-조건부 메시지) father_of+father_of⇒
> grandfather_of 같은 합성을 직접 겨냥. `pair_graph_out`이 zero-init이라 꺼진 상태(이전
> 버전)와 zero-init parity 직접 검증 완료 — 상세는 `Scripts/dk_gat/README.md`.
>
> 이전 (`dk_gat` = Gated Fusion + Bilinear Classifier 실행 완료 확인 —
> **여전히 baseline보다 낮고, 개선폭도 스크리닝 기대치에 훨씬 못 미침**): 이 세션에서 셀 5가
> 이미 GPU(RTX PRO 6000)로 끝까지 돌아 있었음. **결과: dev F1 60.62 / Ign F1 58.69 (best
> epoch 6, patience=3으로 epoch 9에서 조기 종료)** — baseline(61.71/59.86)보다 −1.09/−1.17
> 낮고, 직전 heterogeneous-graph-only 버전(60.29/58.39)보다는 **겨우 +0.33/+0.30**만 오름.
> distant 스크리닝의 Gated Fusion(+2.2)·Bilinear(+3.67)를 단순히 더하면 +5.9를 기대했겠지만
> 실제론 1/10도 안 됨 — PU loss 선례(스크리닝 +2.93 → 실측 +0.35)보다도 더 크게 줄어든 사례.
> **이 프로젝트에서 두 번째로 확인된 패턴**: distant-only 1-epoch 스크리닝 점수는 이 아키텍처
> 계열에서 최종 성능의 신뢰할 만한 예측 지표가 아님 — 방향(+/-)만 참고. 상세 근거는
> `Scripts/dk_gat/README.md` 참고.
>
> **이번에 추가로 구현한 다음 시도** (`dk_gat_v2`, 아래 셀 4 커맨드): Entity-Pair Graph
> (`--use_pair_graph`, 위 참고), Meta-path Attention (`--use_metapath_attention`, 엣지
> 카테고리별 독립 attention 벡터), Curriculum PU-weight (`--curriculum_na_weight
> --na_weight_start 1.0`, distant 단계 na_weight를 1.0→0.7로 step 단위 anneal), 학습전략
> 플래그 3개 실전 반영(`--layerwise_lr_decay 0.9`, `--freeze_encoder_epochs 1`,
> `--evidence_start_epoch 2`). CPU smoke test(전 플래그 동시 적용, 크래시 없음)만 통과했고
> **distant 스크리닝은 생략**(시간 제약) — 위 실측 패턴을 감안하면 이번 실행이 확실한 개선을
> 보장하진 않음. **만약 이번에도 baseline을 못 넘으면, GAT 자체가 이 데이터 규모에서 순이익을
> 못 낸다는 뜻일 수 있으므로 ATLOP+PUATLoss(0.7) (62.06/60.16, 이미 baseline 대비 확실한
> 개선)를 주 모델로 놓고 dk EGAT는 "그래프 증강을 시도했으나 이 규모에서는 이득이 없었다"는
> 정직한 ablation 결과로 보고하는 방향을 권장.**
>
> 이전 (모델 다운로드 병목 해결, 셀 구조 정리): Colab ↔ HuggingFace CDN 네트워크가 불안정해서
> `model.safetensors`(436MB) 다운로드가 반복적으로 멈추는 문제가 있었음. **`aria2c`(멀티커넥션 +
> 세그먼트별 자동 재시도)로 `bert-base-cased`를 `/content/bert-base-cased`에 먼저 통째로 받아두고,
> 학습 커맨드에 `--model_name_or_path /content/bert-base-cased`를 추가**.
>
> 이전 (그래프/pair representation 고도화): ATLoss 교체 효과가 실측으로
> 확인됨(distant 프리트레인만 dev F1 46.58/Ign 44.21 — BCE 때 24.77에서 회복, 20k 기준 참고치
> 43.15~47.79 범위 안) → 보류해뒀던 고도화 2건 반영.
>
> ① **Entity-Sentence Heterogeneous Graph**: 그래프 노드에 문장(Sentence)을 추가, entity-entity
> 엣지에 더해 entity-sentence("등장함") 엣지 신설. 직접/같은 문장/멘션겹침으로 안 이어진 두
> 엔티티도 공통 문장 노드를 거쳐 2-hop 만에 정보 교환 가능 (예: Steve Jobs -[S1]- Apple -[S2]-
> California). GAT 통과 후 엔티티 행만 잘라내 pair 구성에 사용.
>
> ② **Pair Representation에 element-wise interaction 추가**: `[g_h ‖ g_t ‖ c_ht]`(2304-d)에
> `g_h*g_t`(768-d) 곱 항을 더해 `[g_h ‖ g_t ‖ g_h*g_t ‖ c_ht]`(3072-d)로 확장.
>
> ③ **best-epoch 체크포인트 저장 추가**: 매 epoch dev F1이 갱신될 때마다 `{run_name}_best.pt`로
> 저장 — 마지막 epoch이 우연히 저점일 때(실측: epoch 6 F1 59.85 → epoch 7 59.57 하락 관측) 최고
> 기록을 놓치지 않기 위함. 최종 epoch 체크포인트(`{run_name}.pt`, baseline과 비교 기준)는 별도 유지.
>
> 이전: 손실함수를 BCE+threshold sweep → Adaptive Thresholding(ATLoss/PUATLoss)으로 교체 (distant
> 단계 PUATLoss na_weight=0.7, annotated는 자동 전환). 파이프라인 상세는 `Scripts/dk_gat/README.md`.

**실행 전**: 런타임 → 런타임 유형 변경 → **A100 GPU**.

**학습 구성 (`dk_gat_v2`)**: distant 20,000개 × **1 epoch** PUATLoss(na_weight 1.0→0.7 curriculum
anneal) 프리트레인 → annotated 3,053개 × **최대 15 epoch**(early stopping patience=3) ATLoss
파인튜닝 (stage1 AdamW 2e-5, stage2 1e-5·layerwise decay 0.9, wd 0.01, warmup 6%, clip 1.0,
seed 66, 각 스테이지 첫 1epoch 인코더 동결, evidence loss는 3번째 epoch부터) + **Gated Fusion +
Grouped Bilinear Classifier + Meta-path Attention + Entity-Pair Graph**. 예측은 Adaptive
Thresholding으로 결정.

**비교 기준** (`results/comparison.md`, 전부 동일 스코어러·seed 66·distant 20k·distant_epochs 1):

| 모델 | annotated epochs | dev F1 | Ign F1 |
|---|---|---|---|
| ATLOP baseline | 15 | 61.71 | 59.86 |
| ATLOP + PUATLoss(0.7) | 15 | 62.06 | 60.16 |
| dk EGAT (엔티티 그래프, ATLoss 교체 직후) | 8/15까지 관측 | 59.85(peak) | 58.12(peak) |
| dk EGAT + Heterogeneous graph + interaction term (JK on, GF/Bilinear 없음) | 15 | 60.29(peak, epoch13) | 58.39(peak) |
| dk EGAT + Gated Fusion + Bilinear Classifier (`run_name=dk_gat`, **실측 완료**) | 15 (early stop @ epoch9) | **60.62**(peak, epoch6) | **58.69**(peak) |
| **dk EGAT v2: + Meta-path Attention + Curriculum PU-weight + Entity-Pair Graph + 학습전략 플래그 (이 실행, `run_name=dk_gat_v2`)** | 15(early stop patience 3) | | |


In [6]:
# 0) GPU 확인 (A100이 보여야 함)
!nvidia-smi -L

GPU 0: NVIDIA RTX PRO 6000 Blackwell Server Edition (UUID: GPU-26f99268-6f9f-55f2-225b-31e300f8d648)


In [7]:
# 1) 코드 + 데이터 (Git LFS에서 필요한 json만 선별 pull)
#    Scripts/dk_gat는 아직 main이 아니라 dk 브랜치에만 있으므로 반드시 checkout 필요
!GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/multiful/Information_Extraction.git
%cd Information_Extraction
!git checkout dk
!git lfs pull --include="docred_data/data/dev.json,docred_data/data/train_annotated.json,docred_data/data/train_distant.json,docred_data/data/rel_info.json"
# json들이 MB~GB 단위로 보이면 성공 (133바이트면 LFS pull 실패)
!ls -lh docred_data/data/*.json
# Scripts/dk_gat가 실제로 있는지 확인 (여기 안 뜨면 아래 학습 셀에서 ModuleNotFoundError 남)
!ls Scripts/dk_gat/

Cloning into 'Information_Extraction'...
remote: Enumerating objects: 483, done.
remote: Counting objects: 100% (242/242), done.
remote: Compressing objects: 100% (160/160), done.
remote: Total 483 (delta 130), reused 177 (delta 82), pack-reused 241 (from 1)
Receiving objects: 100% (483/483), 32.70 MiB | 27.63 MiB/s, done.
Resolving deltas: 100% (238/238), done.
/content/Information_Extraction/Information_Extraction
Branch 'dk' set up to track remote branch 'dk' from 'origin'.
Switched to a new branch 'dk'
-rw-r--r-- 1 root root 4.1M Jul 14 13:28 docred_data/data/dev.json
-rw-r--r-- 1 root root  132 Jul 14 13:28 docred_data/data/dev_revised.json
-rw-r--r-- 1 root root 2.4K Jul 14 13:28 docred_data/data/rel_info.json
-rw-r--r-- 1 root root  132 Jul 14 13:28 docred_data/data/test.json
-rw-r--r-- 1 root root  132 Jul 14 13:28 docred_data/data/test_revised.json
-rw-r--r-- 1 root root  13M Jul 14 13:28 docred_data/data/train_annotated.json
-rw-r--r-- 1 root root 417M Jul 14 13:29 docred_dat

In [8]:
# 2) 패키지 (모델은 허브가 아니라 로컬 파일에서 로드할 거라 aria2도 여기서 같이 설치)
!pip install -q transformers==4.57.6 accelerate
!apt-get -qq install -y aria2 > /dev/null

In [9]:
%%bash
# 3) bert-base-cased를 로컬로 직접 받기 (허브 다운로더 대신 aria2c 사용)
#    Colab <-> HF CDN 네트워크가 불안정해서 순정 다운로더는 느리게라도 버텼지만(최저 158kB/s),
#    hf_transfer는 세그먼트 하나가 끊기면 재시도 없이 그대로 멈춰버리는 것으로 확인됨
#    (65MB/s로 빠르게 가다 15%에서 하드행). aria2c는 -x/-s로 멀티커넥션, 끊기면 그 세그먼트만
#    재시도(--max-tries, --retry-wait)하므로 이 네트워크 환경에서 훨씬 안정적.
set -e
mkdir -p /content/bert-base-cased
BASE_URL="https://huggingface.co/bert-base-cased/resolve/main"
for fname in model.safetensors config.json vocab.txt tokenizer_config.json tokenizer.json; do
  aria2c -x 16 -s 16 -k 1M --max-tries=20 --retry-wait=2 --continue=true \
    -d /content/bert-base-cased -o "$fname" \
    "$BASE_URL/$fname"
done
echo "--- 받은 파일 확인 (model.safetensors가 436M 근처여야 정상, 133바이트면 실패) ---"
ls -lh /content/bert-base-cased


07/14 13:30:02 [NOTICE] Downloading 1 item(s)

07/14 13:30:02 [NOTICE] CUID#7 - Redirecting to https://cas-bridge.xethub.hf.co/xet-bridge-us/621ffdc036468d709f174331/83c31be240458b001866527feebc3cece210a4aec957064b2f166d2dd6e8471f?Expires=1784039402&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly9jYXMtYnJpZGdlLnhldGh1Yi5oZi5jby94ZXQtYnJpZGdlLXVzLzYyMWZmZGMwMzY0NjhkNzA5ZjE3NDMzMS84M2MzMWJlMjQwNDU4YjAwMTg2NjUyN2ZlZWJjM2NlY2UyMTBhNGFlYzk1NzA2NGIyZjE2NmQyZGQ2ZTg0NzFmKiIsIkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc4NDAzOTQwMn19fV19&Signature=MEUCIQC1N%7EbpYph2yfZxfmpvt4M2nIzGkiksBC0hXKnrg0nl8wIgM3hQwNEnSNpkn5Rv7SnBFigrFHqKNGniG%7EVfq3O6Z48_&Key-Pair-Id=K3EPXBYC3CKDRZ&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27model.safetensors%3B+filename%3D%22model.safetensors%22%3B&X-Xet-Cas-Uid=public&X-Amz-Algorithm=AWS4-HMAC-SHA256&X-Amz-Credential=cas%2F20260714%2Fus-east-1%2Fs3%2Faws4_request&X-Amz-Date=20260714T133002Z&X-Amz-Expires=3600&X-Amz-SignedHeade

In [10]:
# 3.5) 이미 완료된 dk_gat(Gated Fusion+Bilinear only) 실행 결과 먼저 백업
#    (dev F1 60.62/Ign F1 58.69, best epoch 6 -- baseline보다 낮지만 아직 Drive에 백업 전이라
#    세션 끊기면 사라짐. v2 재학습 전에 반드시 먼저 실행)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1"
!cp results/dk_gat.pt results/dk_gat_best.pt \
   results/dk_gat_stage1.pt results/dk_gat_stage1_best.pt \
   results/dk_gat_dev_predictions.json results/dk_gat_best_dev_predictions.json \
   "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1/"
!ls -lh "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1/" | grep dk_gat

Mounted at /content/drive
cp: cannot stat 'results/dk_gat.pt': No such file or directory
cp: cannot stat 'results/dk_gat_best.pt': No such file or directory
cp: cannot stat 'results/dk_gat_stage1.pt': No such file or directory
cp: cannot stat 'results/dk_gat_stage1_best.pt': No such file or directory
cp: cannot stat 'results/dk_gat_dev_predictions.json': No such file or directory
cp: cannot stat 'results/dk_gat_best_dev_predictions.json': No such file or directory
-rw------- 1 root root 773K Jul 14 05:49 dk_gat_best_dev_predictions.json
-rw------- 1 root root 430M Jul 14 05:49 dk_gat_best.pt
-rw------- 1 root root 759K Jul 14 05:49 dk_gat_dev_predictions.json
-rw------- 1 root root 430M Jul 14 05:49 dk_gat.pt
-rw------- 1 root root 430M Jul 14 05:49 dk_gat_stage1_best.pt
-rw------- 1 root root 430M Jul 14 05:49 dk_gat_stage1.pt


In [11]:
# 4) 학습 v2: distant 20k×1ep -> annotated 15ep (baseline과 동일 스케줄, GPU 약 2~3시간)
#    이전 dk_gat(Gated Fusion+Bilinear only) 실행은 이미 완료됨 -- dev F1 60.62/Ign F1 58.69
#    (best epoch 6), baseline(61.71/59.86)보다 여전히 낮음. run_name을 dk_gat_v2로 바꿔서
#    이전 실행 결과(results/dk_gat*.pt)를 덮어쓰지 않고 남겨둠 -- 백업 전이면 아래 5)번 셀을
#    먼저 실행할 것.
#    --use_pair_graph --pair_graph_dim 256: (h,t) 쌍 자체를 노드로 하는 2차 그래프 --
#    same-head/same-tail/bridge-succ/bridge-pred 4타입 방향성 엣지 + 이 쌍의 provisional
#    관계 예측(logits)을 노드 특징에 얹는 관계-조건부 메시지로 A->B,B->C=>A->C 합성을 직접
#    겨냥 (기존 Entity+Sentence GAT는 엔티티 임베딩만 보강할 뿐 이 경로가 없었음). zero-init
#    residual이라 꺼진 상태와 zero-init parity 검증 완료 -- 신규, distant 스크리닝 안 됨,
#    CPU smoke test만 통과.
#    --use_pu_loss --na_weight 0.7 --curriculum_na_weight --na_weight_start 1.0: distant 단계
#    PUATLoss na_weight를 step마다 1.0(=사실상 일반 ATLoss, 초반엔 distant 레이블 전부 신뢰)에서
#    0.7(기존 검증값)까지 선형 anneal -- 신규, distant 스크리닝 안 됨, CPU smoke test만 통과.
#    --use_gated_fusion --use_bilinear_classifier: 이미 distant A/B로 검증된 조합이지만, 실측
#    결과(dev F1 60.62)는 스크리닝 기대치(+2.2/+3.67)에 훨씬 못 미쳤음 -- README.md 참고.
#    --use_metapath_attention: GAT attention 파라미터를 엣지 카테고리별 독립 벡터로 분리 -- 신규,
#    distant 스크리닝 안 됨, CPU smoke test만 통과.
#    --lr2 1e-5 --layerwise_lr_decay 0.9 --freeze_encoder_epochs 1 --evidence_start_epoch 2
#    --early_stop_patience 3: 학습전략 플래그. lr2/early_stop_patience는 이전 사이클에 이미
#    반영됐고, layerwise_lr_decay/freeze_encoder_epochs는 이번에 처음 반영(신규, CPU smoke
#    test만 통과).
!python -m Scripts.dk_gat.train_gat \
  --model_name_or_path /content/bert-base-cased \
  --distant_limit 20000 --distant_epochs 1 --epochs 15 \
  --use_pu_loss --na_weight 0.7 --curriculum_na_weight --na_weight_start 1.0 \
  --use_gated_fusion --use_bilinear_classifier --use_metapath_attention \
  --use_pair_graph --pair_graph_dim 256 \
  --lr 2e-5 --lr2 1e-5 --layerwise_lr_decay 0.9 --freeze_encoder_epochs 1 \
  --evidence_start_epoch 2 --early_stop_patience 3 \
  --weight_decay 0.01 --warmup_ratio 0.06 --dropout 0.1 \
  --run_name dk_gat_v2 --save_model --seed 66

[device] cuda
[data] train=3053 dev=998 docs
preprocess-gat: 100% 998/998 [00:02<00:00, 341.72it/s]
preprocess-gat: 100% 3053/3053 [00:08<00:00, 347.64it/s]
2026-07-14 13:32:36.411861: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-14 13:32:36.444774: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[stage 1] loss = PUATLoss(na_weight=0.7)
[stage 1] distant pretrain on 20000 docs (1 epoch(s))
preprocess-gat: 100% 20000/20000 [00:54<00:00, 368.51it/s]
  [distant-pretrain] encoder frozen (epoch

: 

In [ ]:
# 5) v2 결과 백업 (세션 끊기면 파일이 사라지므로 반드시 실행)
#    best_* = dev F1 최고 epoch 체크포인트/예측 (마지막 epoch보다 나을 수 있음, 둘 다 백업)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1"
!cp results/dk_gat_v2.pt results/dk_gat_v2_best.pt \
   results/dk_gat_v2_stage1.pt results/dk_gat_v2_stage1_best.pt \
   results/dk_gat_v2_dev_predictions.json results/dk_gat_v2_best_dev_predictions.json \
   "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1/"
!ls -lh "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1/" | grep dk_gat_v2

## 결과 기록

마지막 epoch 로그 수치를 위의 비교표와 `results/comparison.md`에 반영. 로그에 `<- new best`가
찍힌 마지막 줄이 `dk_gat_v2_best.pt`에 저장된 epoch — 최종 epoch 수치와 다르면 (더 낮으면) best
쪽 수치도 같이 기록해둘 것.

- stage-1(distant 직후) 수치도 따로 적어두면 GAT가 노이즈 라벨 위에서 얼마나 버티는지,
  annotated 파인튜닝이 얼마나 끌어올리는지 분해해 보여줄 수 있음.
- seed 66 단일 시드 — baseline과의 차이가 ±1점 이내면 PRD §5 시드 노이즈 주의 적용.
- Evidence Contrastive Loss는 evidence_start_epoch(=2)부터 실질 작동함(distant는 evidence 없음).
- GAT 고도화 추가 실험(edge feature ablation, 헤드 수, 인접행렬 반경 등)은 이 노트북의 셀 4에
  `--gat_heads`, `--evidence_weight` 인자만 바꿔 반복하면 됨.
- **이번 v2 실행이 baseline(61.71/59.86)을 못 넘으면**: Entity-Pair Graph/Meta-path
  Attention/Curriculum PU-weight/layerwise_lr_decay/freeze_encoder_epochs 중 어느 게 원인인지
  이 한 번의 실행만으론 분리 안 됨(전부 처음으로 한꺼번에 켰기 때문). 그래도 `--use_pair_graph`는
  zero-init 잔차라 "꺼도 최소한 손해는 없는" 성질이라, 시간이 있다면 이것부터 단독으로
  A/B(`--use_pair_graph` 있음/없음, 나머지 동일)해보는 게 가장 저비용으로 원인을 좁히는 길.
  추가로 스크리닝할 시간이 없다면, dk EGAT는 baseline 대비 순이익이 없었다는 결과로 report에
  정직하게 기록하고 ATLOP+PUATLoss(0.7)(62.06/60.16)를 주 모델로 제시하는 게 안전한 선택 —
  `Scripts/dk_gat/README.md` 최상단 참고.